# Combining Options

## Import packages and define quantum platform

In [ ]:
import numpy as np
from laboneq._automation.utils import load_automation_parameters
from laboneq.simple import *

from laboneq_applications._automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.experiments import amplitude_rabi, ramsey
from laboneq_applications.experiments.options import TuneUpWorkflowOptions
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

In [ ]:
# Create a demonstration QuantumPlatform for a tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=4)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup:
setup = qt_platform.setup

# And a tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of six LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

session = Session(setup)
session.connect(do_emulation=True)

## Running experiment workflows manually

### Amplitude Rabi

In [ ]:
opts = TuneUpWorkflowOptions()
opts.evaluate = False
opts.update = True
opts

In [ ]:
experiment_workflow = amplitude_rabi.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=qubits[0],
    amplitudes=np.linspace(0, 1, 11),
    options=opts,
)
workflow_result = experiment_workflow.run()

In [ ]:
type(workflow_result)

In [ ]:
workflow_result.tasks["analysis_workflow"].output

In [ ]:
analysis_results = workflow_result.tasks["analysis_workflow"]
fit_results = analysis_results.tasks["fit_data"].output
fit_results["q0"].rsquared

In [ ]:
experiment_workflow.graph.tree

### Ramsey

In [ ]:
opts = TuneUpWorkflowOptions()
opts.evaluate = True
opts.update = True
opts

In [ ]:
experiment_workflow = ramsey.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=qubits[0],
    delays=np.linspace(0, 20e-6, 51),
    detunings=0.67e6,
    options=opts,
)
workflow_result = experiment_workflow.run()

In [ ]:
workflow_result.tasks["analysis_workflow"].output

In [ ]:
analysis_results = workflow_result.tasks["analysis_workflow"]
fit_results = analysis_results.tasks["fit_data"].output
fit_results["q0"].rsquared

In [ ]:
experiment_workflow.graph.tree

## Running experiment workflows using the automation framework

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters("combining_options.yml"),
)

In [ ]:
r1 = WorkflowLayer(
    ramsey.experiment_workflow, ["q0", "q1", "q2"], key="r1", depends_on=["__root__"]
)
auto.add_layer(r1)
r2 = WorkflowLayer(
    ramsey.experiment_workflow, ["q0", "q1", "q2"], key="r2", depends_on=["r1"]
)
auto.add_layer(r2)
auto.plot();

In [ ]:
auto.run_layer("r1")
auto.plot();

In [ ]:
auto.get_layer("r1").status

In [ ]:
auto.deactivate_node("r1_q0")
auto.plot();

In [ ]:
auto.get_layer("r1").status

In [ ]:
auto.run_layer("r1")

In [ ]:
auto.reset()
auto.plot();

In [ ]:
auto.get_layer("r1").quantum_elements

In [ ]:
auto.get_layer("r1").sequential = True

In [ ]:
auto.run_layer("r1")
auto.plot();

In [ ]:
auto.get_layer("r1").workflow_results

### Combining active reset

In [ ]:
auto.get_layer("r1").workflow_options.active_reset

In [ ]:
for node in auto.nodes(layer_key="r1"):
    if node is not None and node.workflow_options is not None:
        print(node.workflow_options.active_reset)

In [ ]:
auto.get_node("r1_q0").workflow_options.active_reset(False)

In [ ]:
for node in auto.nodes(layer_key="r1"):
    if node is not None and node.workflow_options is not None:
        print(node.workflow_options.active_reset)

In [ ]:
auto.get_layer("r1").workflow_options.active_reset

### Combining active reset repetitions

In [ ]:
auto.get_layer("r1").workflow_options.active_reset_repetitions

In [ ]:
for node in auto.nodes(layer_key="r1"):
    if node is not None and node.workflow_options is not None:
        print(node.workflow_options.active_reset_repetitions)

In [ ]:
auto.get_node("r1_q0").workflow_options.active_reset_repetitions(5)

In [ ]:
for node in auto.nodes(layer_key="r1"):
    if node is not None and node.workflow_options is not None:
        print(node.workflow_options.active_reset_repetitions)

In [ ]:
auto.get_layer("r1").workflow_options.active_reset_repetitions

### Combining count

In [ ]:
auto.get_layer("r1").workflow_options.count

In [ ]:
for node in auto.nodes(layer_key="r1"):
    if node is not None and node.workflow_options is not None:
        print(node.workflow_options.count)

In [ ]:
auto.get_node("r1_q0").workflow_options.count(4096)

In [ ]:
for node in auto.nodes(layer_key="r1"):
    if node is not None and node.workflow_options is not None:
        print(node.workflow_options.count)

In [ ]:
auto.get_layer("r1").workflow_options.count